In [1]:
import mlflow
# Step 1: Set up the MLflow tracking server
mlflow.set_tracking_uri("http://184.72.71.39:5000/")

c:\Users\Abhi\Documents\youtube-comment-analysis\mlops-youtube-comment-analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Set or create an experiment
mlflow.set_experiment("ML Algos with HP Tuning")

2026/09/09 15:26:25 INFO mlflow.tracking.fluent: Experiment with name 'ML Algos with HP Tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://comment-analysis-bucket-994/6', creation_time=1788963985769, effective_trace_archival_retention=None, experiment_id='6', last_update_time=1788963985769, lifecycle_stage='active', name='ML Algos with HP Tuning', tags={}, trace_location=None, workspace='default'>

In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier
import mlflow
import mlflow.sklearn
import optuna

In [4]:
df = pd.read_csv('../data/processed/processed_comments.csv').dropna()
df.shape

(36662, 2)

In [5]:
# Remap labels
df['category'] = df['category'].replace({-1: 2})
df = df.dropna(subset=['category', 'clean_comment'])

ngram_range = (1, 3)
max_features = 1000

# Split the data first
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_comment'],
    df['category'],
    test_size=0.2,
    random_state=42,
    stratify=df['category']
)

# Separate validation set for Optuna
X_train_inner, X_val, y_train_inner, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

# TF-IDF on training data only
vectorizer = TfidfVectorizer(
    ngram_range=ngram_range,
    max_features=max_features
)

X_train_inner_vec = vectorizer.fit_transform(X_train_inner)
X_val_vec = vectorizer.transform(X_val)

# SMOTE only on training data
smote = SMOTE(random_state=42)
X_train_inner_vec, y_train_inner = smote.fit_resample(
    X_train_inner_vec,
    y_train_inner
)


# Optuna objective
def objective_lightgbm(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    learning_rate = trial.suggest_float(
        'learning_rate',
        1e-4,
        1e-1,
        log=True
    )
    max_depth = trial.suggest_int('max_depth', 3, 10)

    model = LGBMClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        random_state=42,
        verbosity=-1
    )

    model.fit(X_train_inner_vec, y_train_inner)

    y_pred = model.predict(X_val_vec)

    return accuracy_score(y_val, y_pred)


# Run Optuna
study = optuna.create_study(direction="maximize")
study.optimize(objective_lightgbm, n_trials=20)

best_params = study.best_params

print("Best parameters:", best_params)
print("Best validation accuracy:", study.best_value)


# Train final model using full training data
final_vectorizer = TfidfVectorizer(
    ngram_range=ngram_range,
    max_features=max_features
)

X_train_vec = final_vectorizer.fit_transform(X_train)
X_test_vec = final_vectorizer.transform(X_test)

smote = SMOTE(random_state=42)
X_train_vec, y_train = smote.fit_resample(
    X_train_vec,
    y_train
)

best_model = LGBMClassifier(
    n_estimators=best_params['n_estimators'],
    learning_rate=best_params['learning_rate'],
    max_depth=best_params['max_depth'],
    random_state=42,
    verbosity=-1
)

best_model.fit(X_train_vec, y_train)

y_pred = best_model.predict(X_test_vec)

accuracy = accuracy_score(y_test, y_pred)

print("Final accuracy:", accuracy)


# Log results in MLflow
with mlflow.start_run():

    mlflow.set_tag(
        "mlflow.runName",
        "LightGBM_SMOTE_TFIDF_Trigrams"
    )

    mlflow.set_tag(
        "experiment_type",
        "algorithm_comparison"
    )

    mlflow.log_param("algo_name", "LightGBM")
    mlflow.log_param("ngram_range", str(ngram_range))
    mlflow.log_param("max_features", max_features)
    mlflow.log_param("n_trials", 20)

    mlflow.log_params(best_params)

    mlflow.log_metric("accuracy", accuracy)

    classification_rep = classification_report(
        y_test,
        y_pred,
        output_dict=True
    )

    for label, metrics in classification_rep.items():
        if isinstance(metrics, dict):
            for metric, value in metrics.items():
                mlflow.log_metric(
                    f"{label}_{metric}",
                    value
                )

    mlflow.lightgbm.log_model(
        best_model,
        name="LightGBM_model"
    )

[I 2026-09-09 15:33:55,111] A new study created in memory with name: no-name-d3700db4-43ef-40b6-8e96-382ad70137a7
[I 2026-09-09 15:33:59,375] Trial 0 finished with value: 0.7604841459256734 and parameters: {'n_estimators': 134, 'learning_rate': 0.0871849904903694, 'max_depth': 5}. Best is trial 0 with value: 0.7604841459256734.
[I 2026-09-09 15:34:03,426] Trial 1 finished with value: 0.6905898397545176 and parameters: {'n_estimators': 144, 'learning_rate': 0.02511258369846798, 'max_depth': 5}. Best is trial 0 with value: 0.7604841459256734.
[I 2026-09-09 15:34:05,241] Trial 2 finished with value: 0.49727241732015004 and parameters: {'n_estimators': 213, 'learning_rate': 0.00014615148307720152, 'max_depth': 3}. Best is trial 0 with value: 0.7604841459256734.
[I 2026-09-09 15:34:07,033] Trial 3 finished with value: 0.5719399931810433 and parameters: {'n_estimators': 72, 'learning_rate': 0.0001079700400589588, 'max_depth': 7}. Best is trial 0 with value: 0.7604841459256734.
[I 2026-09-09 

Best parameters: {'n_estimators': 287, 'learning_rate': 0.05648415598090134, 'max_depth': 9}
Best validation accuracy: 0.7913399249914763
Final accuracy: 0.7837174417018955
🏃 View run LightGBM_SMOTE_TFIDF_Trigrams at: http://184.72.71.39:5000/#/experiments/6/runs/cf4626231db94915bfdc4f4ef754fc04
🧪 View experiment at: http://184.72.71.39:5000/#/experiments/6
